# Module-Level Depression–Cognition Association Analysis

---

## Purpose

This notebook executes the end-to-end analysis that compares cognitive performance between:
- a **depression cohort** and a **control reference cohort**, and
- **module-level connectivity-derived clusters** within the depression cohort against controls and each other,

using **robust (median/MAD) z-scoring** and **quantile regression** (R/`quantreg` via `rpy2`).

It follows the structure of `modular_cog_associations_main.py` step-by-step and delegates statistical work and plotting to helpers in `modular_cog_associations_utils.py`.

---

## High-Level Pipeline

| Step | Label | What happens |
|------|-------|-------------|
| **1** | R environment | Load R packages (`quantreg`), configure rpy2 sinks so R console output goes to a log file |
| **2** | Load & preprocess data | Load the module-level cohort CSV via `load_and_rename_cohort_data`; verify cohort sizes |
| **3** | Task-wise robust z-scores | Compute median/MAD z-scores (referenced to controls) for every cognitive task; run one-sample median tests via quantile regression; apply FDR correction |
| **4** | Domain composite z-scores | Aggregate task z-scores into 6 cognitive domains (median across tasks); test each domain against zero; apply FDR correction |
| **5** | Visualizations (overall cohort) | Radar/violin plots of task and domain z-score profiles for the full depression cohort |
| **6** | Per-cluster analysis | Repeat Steps 3–5 for each connectivity modality × direction × cluster; run between-cluster quantile regression; register significance for radar overlay |

---

## Inputs

### Primary input file
| File | Description |
|------|-------------|
| `data/UKB/cohorts/module_connectivity_features_with_covariates.csv` | Main input table — must contain `eid`, `depression_status` (0=control, 1=depressed), cognitive task columns (added from combined cohort if missing), and cluster assignment columns like `functional_internal_cluster` |

### Optional file (used only to add missing task columns)
| File | Description |
|------|-------------|
| `data/UKB/cohorts/combined_cohort_<DEPRESSION_CODES>.csv` | Combined cohort table; needed only when cognitive task UKB field columns are missing from the main input |

### Column requirements
- `depression_status`: int 0 (control) or 1 (depressed)
- `<conn_type>_<dir>_cluster` columns (e.g. `functional_internal_cluster`): values `'0'` or `'1'` for depressed subjects; absent or NaN for controls
- Covariate columns: `age_at_assessment`, `sex`, `I10`, `Z864`, `F419`

---

## Configurable Constants

| Constant | Default | Description |
|----------|---------|-------------|
| `DEPRESSION` | `["F32"]` | ICD10 code(s) selecting the depression cohort |
| `COVARIATES` | `["age_at_assessment","sex","I10","Z864","F419"]` | Covariates for all quantile regression models |
| `DOMAINS` | 6-domain dict | Mapping from domain name → list of task column names |
| `PLOTS_DIR` | `"reports/plots"` | Output directory for all figures |
| `OUTPUT_DIR` | `"data/UKB/cohorts"` | Output directory for logs and CSV |
| `DATA_DIR` | `"data/UKB/cohorts"` | Input data directory |
| `DATA_FILE` | `"module_connectivity_features_with_covariates.csv"` | Main input CSV file name (must be in `DATA_DIR`) |
| `QUANTILE_REGRESSION_BOOTSTRAP_R` | `1000` | Number of bootstrap resamples for quantile regression |

---

## Outputs

### Figures (`reports/plots/`)
| Pattern | Description |
|---------|-------------|
| `<CODES>_cognition_task_z_scores.png` | Overall cohort task z-score radar |
| `<CODES>_cognition_domain_z_scores.png` | Overall cohort domain z-score radar |
| `schaefer1000+tian54/<conn>_con/modular_..._task_z_scores_<Cluster_N>_<dir>.png` | Per-cluster task z-score radars |
| `schaefer1000+tian54/<conn>_con/modular_..._domain_z_scores_<Cluster_N>_<dir>.png` | Per-cluster domain z-score radars |
| `schaefer1000+tian54/<conn>_con/modular_..._task_clusters_violin_<dir>.png` | Task z-score violin plots by cluster |
| `schaefer1000+tian54/<conn>_con/modular_..._domain_clusters_violin_<dir>.png` | Domain composite violin plots by cluster |
| `schaefer1000+tian54/<conn>_con/modular_<mod>_<dir>_..._task_connectivity.png` | Per-module connectivity vs task z-scores scatter |
| `schaefer1000+tian54/<conn>_con/modular_<mod>_<dir>_..._domain_connectivity.png` | Per-module connectivity vs domain z-scores scatter |

### CSV
| File | Description |
|------|-------------|
| `data/UKB/cohorts/depression_cohort_z_scores_<CODES>.csv` | Depression cohort with all task and domain z-score columns |

### Text logs (`data/UKB/cohorts/`)
| File | Description |
|------|-------------|
| `modular_R_analysis_summary_<CODES>.txt` | R console output and model summaries from every QR call |
| `modular_multiple_testing_summary_<CODES>.txt` | FDR correction tables (raw + adjusted p-values, significance) |
| `modular_robust_z_scores_summary_<CODES>.txt` | Control reference statistics (medians, MADs) and depression z-score summaries |
| `modular_composite_z_scores_summary_<CODES>.txt` | Domain composite z-score statistics |

---

## Key Helper Functions (from `modular_cog_associations_utils`)

| Function | Role |
|----------|------|
| `setup_r_environment()` | Install/load R packages, configure rpy2 |
| `load_and_rename_cohort_data(path)` | Load CSV and canonicalize column names from UKB field IDs |
| `calculate_robust_z_scores(data, vars, …)` | Compute median/MAD referenced z-scores; write log |
| `calculate_composite_z_score(data, z_vars, …)` | Aggregate task z-scores into a domain composite; write log |
| `quantile_regression(tmp_csv, dep_vars, …)` | R `quantreg::rq` wrapper; returns dict of p-values |
| `apply_multiple_testing_correction(p_values, …)` | FDR/Bonferroni correction; write formatted log table |
| `plot_z_scores(data, z_vars, …)` | Radar/violin plots of z-score profiles; saves PNG |
| `plot_cognitive_distributions_violin(data, vars, …)` | Violin distributions per group/cluster; saves PNG |
| `plot_conn_cognition_association(data, …)` | Scatter of connectivity vs cognitive z-scores; saves PNG |
| `register_radar_overlay_significance(…)` | Cache FDR p-values into registry for multi-cluster radar overlays |

---

## Performance Notes
- Quantile regression with `R=1000` bootstrap resamples is **CPU-intensive**. The first run of Step 3 and Step 4 may each take several minutes, and the per-cluster loop (Step 6) multiplies this by 6 modality × direction combinations × 2 clusters.
- To test the pipeline quickly, reduce `R` (e.g. `R=10`) in the quantile regression calls below.
- R and the `quantreg` package must be installed and accessible from your Python environment via `rpy2`.

## Imports and Configuration

All imports come from `modular_cog_associations_utils`, which provides every statistical helper, R interface, and plotting function used by this pipeline. The constants below control which cohort is analysed, which covariates enter every regression model, and where outputs are written.

**Cognitive domains** (`DOMAINS`) define how individual task columns are grouped into six higher-order cognitive constructs. Each domain gets its own composite z-score (Step 4).

In [ ]:
import os
import sys
import pandas as pd

# Make the source_code/plots folder importable (adjust if running from a
# different working directory)
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "source_code", "clinical_associations"))
# Alternatively, if the notebook is inside /notebooks and the repo root is one
# level up, use:
# sys.path.insert(0, os.path.join("..", "source_code", "plots"))

from module_cog_associations_utils import (
    setup_r_environment,
    quantile_regression,
    load_and_rename_cohort_data,
    plot_conn_cognition_association,
    apply_multiple_testing_correction,
    calculate_robust_z_scores,
    calculate_composite_z_score,
    plot_z_scores,
    plot_cognitive_distributions_violin,
    register_radar_overlay_significance,
)

# ---------------------------------------------------------------------------
# Configuration — edit these to change cohort, covariates, or file locations
# ---------------------------------------------------------------------------

ASSOCIATION_with = "cognition"          # analysis target label used in plot filenames
COVARIATES = [
    "age_at_assessment", "sex",         # demographic covariates
    "I10", "Z864", "F419"               # ICD comorbidity / medication covariates
]

DEPRESSION       = ["F32"]             # ICD10 code(s) that define the depression cohort
DEPRESSION_CODES = "_".join(DEPRESSION) # e.g. "F32"; used for all output filenames

PLOTS_DIR  = ".../reports/plots"         # output directory for all figures
OUTPUT_DIR = ".../data/UKB/cohorts"      # output directory for logs and CSV
DATA_DIR   = ".../data/UKB/cohorts"      # location of input CSVs
DATA_FILE  = os.path.join(DATA_DIR, "module_connectivity_features_with_covariates.csv")

os.makedirs(PLOTS_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Bootstrap iterations for quantile regression (R) standard errors and confidence intervals
QUANTILE_REGRESSION_BOOTSTRAP_R = 1000

# ---------------------------------------------------------------------------
# Cognitive domains — each maps a human-readable label -> list of task columns
# (the task columns must exist in DATA_FILE after Step 0)
# ---------------------------------------------------------------------------
DOMAINS = {
    "processing_speed": [
        "Snap_task_mean_reaction_time",
        "Symbol_digit_substitution_task_correct",
        "Trail_making_A_duration",
        "Trail_making_B_duration",
    ],
    "working_memory": [
        "Reverse_number_recall_task_span",
        "Pairs_matching_task_errors_6_pairs",
    ],
    "executive_function": [
        "Trail_making_B_duration",
        "Tower_rearranging_task_correct",
        "Snap_task_mean_reaction_time",
    ],
    "associative_learning": [
        "Paired_associates_learning_task_correct",
    ],
    "abstract_reasoning": [
        "Matrix_pattern_completion_correct",
        "Fluid_intelligence_score",
    ],
    "verbal_ability": [
        "Vocabulary_score",
    ],
}

# Variables where higher raw score = worse performance → z-scores are reversed
VARS_TO_REVERSE = [
    "Snap_task_mean_reaction_time",
    "Trail_making_A_duration",
    "Trail_making_B_duration",
    "Pairs_matching_task_errors_6_pairs",
]

# Log file paths
R_LOG_PATH          = os.path.join(OUTPUT_DIR, f"modular_R_analysis_summary_{DEPRESSION_CODES}.txt")
MT_LOG_PATH         = os.path.join(OUTPUT_DIR, f"modular_multiple_testing_summary_{DEPRESSION_CODES}.txt")
ROBUST_Z_LOG_PATH   = os.path.join(OUTPUT_DIR, f"modular_robust_z_scores_summary_{DEPRESSION_CODES}.txt")
COMPOSITE_Z_LOG_PATH= os.path.join(OUTPUT_DIR, f"modular_composite_z_scores_summary_{DEPRESSION_CODES}.txt")

GROUP_COLUMN = "depression_status"   # 0 = control, 1 = depressed

print("Configuration loaded.")
print(f"  Depression cohort codes : {DEPRESSION_CODES}")
print(f"  Covariates              : {COVARIATES}")
print(f"  Domains                 : {list(DOMAINS.keys())}")
print(f"  Data file               : {DATA_FILE}")
print(f"  Plots directory         : {PLOTS_DIR}")

## Step 0: Ensure Cognitive Task Columns Exist

The module-level input file (`module_connectivity_features_with_covariates.csv`) may have been built from connectivity features alone and therefore lack the UK Biobank cognitive task columns. This step detects missing task columns and, if needed, merges them from a broader combined-cohort CSV by joining on `eid`.

The raw UKB field IDs involved are:

| UKB field ID | Renamed to | Cognitive construct |
|---|---|---|
| `p20023_i2` | `Snap_task_mean_reaction_time` | Reaction time (processing speed) |
| `p23324_i2` | `Symbol_digit_substitution_task_correct` | SDST (processing speed) |
| `p6348_i2` | `Trail_making_A_duration` | TMT-A (processing speed) |
| `p6350_i2` | `Trail_making_B_duration` | TMT-B (executive function / processing speed) |
| `p4282_i2` | `Reverse_number_recall_task_span` | Working memory span |
| `p399_i2_a2` | `Pairs_matching_task_errors_6_pairs` | Associative memory errors |
| `p21004_i2` | `Tower_rearranging_task_correct` | Tower rearranging (executive function) |
| `p20197_i2` | `Paired_associates_learning_task_correct` | Paired associates (associative learning) |
| `p6373_i2` | `Matrix_pattern_completion_correct` | Matrix reasoning (abstract reasoning) |
| `p20016_i2` | `Fluid_intelligence_score` | Fluid intelligence (abstract reasoning) |
| `p26302_i2` | `Vocabulary_score` | Vocabulary (verbal ability) |

> **Skip**: If all columns are already present this cell is a no-op.

In [ ]:
# Raw UK Biobank field IDs for all cognitive tasks used in this analysis
selected_task_columns = [
    "p20023_i2",   # Reaction time
    "p23324_i2",   # Symbol digit substitution
    "p6348_i2",    # Trail making A
    "p6350_i2",    # Trail making B
    "p4282_i2",    # Reverse number recall
    "p399_i2_a2",  # Pairs matching (errors)
    "p21004_i2",   # Tower rearranging
    "p20197_i2",   # Paired associates learning
    "p6373_i2",    # Matrix pattern completion
    "p20016_i2",   # Fluid intelligence
    "p26302_i2",   # Vocabulary
]

data_main = pd.read_csv(DATA_FILE)
missing = [c for c in selected_task_columns if c not in data_main.columns]

if not missing:
    print("All cognitive task columns already present — no merge needed.")
else:
    print(f"Missing {len(missing)} task column(s): {missing}")
    combined_csv = os.path.join(DATA_DIR, f"combined_cohort_{DEPRESSION_CODES}.csv")
    if not os.path.exists(combined_csv):
        raise FileNotFoundError(
            f"Combined cohort CSV not found at '{combined_csv}'. "
            "Please ensure the combined cohort file exists before running this step."
        )

    data_tasks = pd.read_csv(combined_csv, usecols=["eid"] + selected_task_columns)

    # Drop columns that already exist in the module file to avoid merge conflicts
    overlap = set(data_main.columns) & set(data_tasks.columns) - {"eid"}
    if overlap:
        data_main = data_main.drop(columns=list(overlap))

    # Left-join to preserve all rows from the module-level file
    data_merged = pd.merge(data_main, data_tasks, on="eid", how="left")
    data_merged.to_csv(DATA_FILE, index=False)
    print(f"Updated '{DATA_FILE}' with {len(selected_task_columns)} task columns.")
    print(f"  Combined shape: {data_merged.shape}")

## Step 1: Initialize R Environment

`setup_r_environment()` does three things:

1. **Package check** — calls `install_r_package_if_missing` for `quantreg` (and any other required packages). If the package is absent it is installed from CRAN via `rpy2`.
2. **rpy2 initialisation** — ensures the R global environment is accessible from Python.
3. **Console sink** — redirects R's console output (`sink`) to `R_LOG_PATH` so all model summaries and warnings produced by subsequent `quantreg::rq` calls are captured in a persistent log rather than printed inline.

> **Requirement:** R must be installed and the `R_HOME` environment variable must be resolvable by `rpy2`. Run `python -c "import rpy2; print(rpy2.__version__)"` to verify the rpy2 installation.

In [ ]:
print("[1/6] Initializing R environment...")
setup_r_environment()
print("R environment ready. Console output will be appended to:")
print(f"  {R_LOG_PATH}")

## Step 2: Load and Preprocess Data

`load_and_rename_cohort_data` reads the module-level CSV and renames raw UK Biobank field-ID columns to human-readable names (e.g., `p20023_i2` → `Snap_task_mean_reaction_time`). It also casts `depression_status` to int and validates that the expected covariate columns are present.

After loading, cohort sizes are printed for:
- All controls vs. all depressed subjects
- Each of the 6 connectivity modality × direction cluster assignments (`functional_internal_cluster`, `functional_external_cluster`, …)

These counts will be embedded in plot titles throughout the analysis.

In [ ]:
print("[2/6] Loading and preprocessing data...")

# Load and canonicalize the module-level dataset
data = load_and_rename_cohort_data(DATA_FILE)

# ---- Cohort size summary ----
n_controls_total  = int((data[GROUP_COLUMN] == 0).sum())
n_depressed_total = int((data[GROUP_COLUMN] == 1).sum())

print(f"\nOverall cohort:")
print(f"  N controls  : {n_controls_total}")
print(f"  N depressed : {n_depressed_total}")

print("\nCluster sizes (depressed subjects only):")
for conn_type in ["functional", "structural", "sfc"]:
    for dir_type in ["internal", "external"]:
        col = f"{conn_type}_{dir_type}_cluster"
        if col in data.columns:
            n0 = int((data[col] == "0").sum())
            n1 = int((data[col] == "1").sum())
            print(f"  {col:45s}  Cluster 0={n0:4d}  Cluster 1={n1:4d}")
        else:
            print(f"  {col:45s}  NOT FOUND in dataset")

print(f"\nDataset shape : {data.shape}")
print(f"Columns       : {list(data.columns[:10])} …")  # show first 10 columns

## Step 3: Task-Wise Robust Z-Scores

### What are robust z-scores?

For each cognitive task variable $x$, the z-score is defined relative to the **control group's** median $\tilde{\mu}$ and MAD (median absolute deviation):

$$z_i = \frac{0.6745 \cdot (x_i - \tilde{\mu})}{\text{MAD}}$$

The constant $0.6745$ makes the MAD-based estimator consistent with the standard deviation under normality.

**All subjects** (controls + depressed) receive z-scores, but the reference statistics ($\tilde{\mu}$ and MAD) are estimated **from controls only**, so a z-score of 0 means "equal to the control median".

### Direction reversal

For latency/error tasks (reaction time, trail-making duration, matching errors), a **higher raw score = worse performance**. Their z-scores are negated so that for all tasks: *positive z = better than controls*.

### Statistical test

A one-sample **quantile regression** at $\tau = 0.5$ (median regression) against zero is run for each task z-score column, adjusted for `COVARIATES`. This tests whether the median z-score of the depressed cohort differs significantly from zero (i.e. differs from the control median). Bootstrapping (`R=1000`) is used for estimation of standard errors and 95% confidence intervals.

All task-level p-values are then jointly corrected for multiple comparisons using **Benjamini–Hochberg FDR** with false discovery rate $\alpha = 0.05$.

In [ ]:
print("[3/6] Calculating task-wise z-scores and testing significance...")

# Deduplicate variables used across domains before z-scoring
unique_z_vars = sorted(set(var for domain_vars in DOMAINS.values() for var in domain_vars))
print(f"Task variables to z-score ({len(unique_z_vars)}): {unique_z_vars}")

# Compute median/MAD-based robust z-scores for the full cohort
z_scored_data = calculate_robust_z_scores(
    data,
    vars=unique_z_vars,
    group_column=GROUP_COLUMN,
    control_value=0,
    depression_value=1,
    log_path=ROBUST_Z_LOG_PATH,
    log_context="Overall cohort; connectivity_type: N/A; cluster: N/A",
)

# Reverse z-scores for latency/error tasks
# (higher raw value = worse performance → negate so positive z = better)
for var in VARS_TO_REVERSE:
    z_scored_data[f"{var}_z"] = -z_scored_data[f"{var}_z"]
print(f"\nReversed z-scores for: {VARS_TO_REVERSE}")

# Save data to a temp CSV for R to read
z_scored_data.to_csv("/tmp/data.csv", index=False)

# One-sample median regression (τ=0.5) for each task z-score,
# testing whether the median differs from 0 (i.e. from the control group)
p_values_vars_z = quantile_regression(
    tmp_csv_path="/tmp/data.csv",
    dependent_variables=[f"{var}_z" for var in unique_z_vars],
    covariates=COVARIATES,
    group_column=None,        # no between-group contrast; testing against 0
    tau=0.5,
    R=QUANTILE_REGRESSION_BOOTSTRAP_R,
    test_against_zero=True,
    return_effects=False,
    r_output_log_path=R_LOG_PATH,
)
print("\nRaw p-values (task z-scores):")
for var, p in p_values_vars_z.items():
    print(f"  {var:55s}  p = {p:.4f}")

# FDR correction across all task-wise tests
apply_multiple_testing_correction(
    p_values=list(p_values_vars_z.values()),
    variable_names=[f"{var}_z" for var in unique_z_vars],
    test_methods=["Quantile Regression"] * len(unique_z_vars),
    method="fdr_bh",
    alpha=0.05,
    log_path=MT_LOG_PATH,
    log_context="Overall cohort; connectivity_type: N/A; cluster: N/A; level: task-wise",
)
print(f"\nFDR results written to: {MT_LOG_PATH}")

## Step 4: Composite Domain Z-Scores

Each cognitive domain is an aggregate of several task z-scores. `calculate_composite_z_score` computes the **median** across task z-scores within each domain and stores the result as a new column named after the domain (e.g., `processing_speed`).

The six domains are:

| Domain | Constituent tasks |
|---|---|
| `processing_speed` | Snap RT, Symbol digit, Trail A, Trail B |
| `working_memory` | Reverse number recall, Pairs matching errors |
| `executive_function` | Trail B, Tower rearranging, Snap RT |
| `associative_learning` | Paired associates learning |
| `abstract_reasoning` | Matrix pattern, Fluid intelligence |
| `verbal_ability` | Vocabulary score |

Note: some tasks (e.g. `Trail_making_B_duration`, `Snap_task_mean_reaction_time`) contribute to **more than one domain**, which is standard in the neuropsychological literature.

After computing composites, a one-sample median regression is run at the domain level (same approach as Step 3), with FDR correction across the six domains.

In [ ]:
print("[4/6] Calculating composite domain z-scores and testing significance...")

# Build a composite z-score column for each cognitive domain
for domain, vars_list in DOMAINS.items():
    z_vars_domain = [f"{var}_z" for var in vars_list]
    z_scored_data = calculate_composite_z_score(
        z_scored_data,
        z_vars=z_vars_domain,
        output_column=domain,   # stored directly as e.g. "processing_speed"
        method="median",
        log_path=COMPOSITE_Z_LOG_PATH,
        log_context=f"Overall cohort; connectivity_type: N/A; cluster: N/A; domain: {domain}",
    )
print(f"Domain composite columns added: {list(DOMAINS.keys())}")

# One-sample median regression for each domain composite
z_scored_data.to_csv("/tmp/data.csv", index=False)
p_values_domains_z = quantile_regression(
    tmp_csv_path="/tmp/data.csv",
    dependent_variables=list(DOMAINS.keys()),
    covariates=COVARIATES,
    group_column=None,
    tau=0.5,
    R=QUANTILE_REGRESSION_BOOTSTRAP_R,
    test_against_zero=True,
    return_effects=False,
    r_output_log_path=R_LOG_PATH,
)
print("\nRaw p-values (domain composites):")
for domain, p in p_values_domains_z.items():
    print(f"  {domain:30s}  p = {p:.4f}")

# FDR correction across all six domains
apply_multiple_testing_correction(
    p_values=list(p_values_domains_z.values()),
    variable_names=list(DOMAINS.keys()),
    test_methods=["Quantile Regression"] * len(DOMAINS),
    method="fdr_bh",
    alpha=0.05,
    log_path=MT_LOG_PATH,
    log_context="Overall cohort; connectivity_type: N/A; cluster: N/A; level: domain-wise",
)
print(f"\nFDR results appended to: {MT_LOG_PATH}")

## Step 5: Visualizations — Overall Cohort

`plot_z_scores` creates summary figures for the full depression cohort (all clusters pooled). Two separate figures are produced:

1. **Task-level radar/violin** — one spoke/panel per cognitive task variable, showing the z-score distribution of the depressed cohort relative to the control median (z = 0 reference line).
2. **Domain-level radar/violin** — one spoke/panel per cognitive domain composite.

Both figures are saved as PNG files under `PLOTS_DIR`. The `type_z_score` argument controls how panel titles and colour palettes are chosen inside the utility function.

The full z-scored dataframe (controls + depressed, all task and domain columns) is also saved as a CSV for downstream use.

In [ ]:
print("[5/6] Creating z-score visualizations (overall cohort)...")

task_z_vars  = [f"{var}_z" for var in unique_z_vars]
domain_vars  = list(DOMAINS.keys())

# ---- Task-wise z-score plot ----
task_plot_path = f"{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_task_z_scores.png"
plot_z_scores(
    z_scored_data,
    z_vars=task_z_vars,
    association_type=ASSOCIATION_with,
    type_z_score="task",
    overall_title=(
        f"Task Z-Scores (Depression Cohort; "
        f"N_control={n_controls_total}, N_depression={n_depressed_total})"
    ),
    save_path=task_plot_path,
)
print(f"  Task z-score plot saved   : {task_plot_path}")

# ---- Domain composite z-score plot ----
domain_plot_path = f"{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_z_scores.png"
plot_z_scores(
    z_scored_data,
    z_vars=domain_vars,
    association_type=ASSOCIATION_with,
    type_z_score="domain",
    overall_title=(
        f"Domain Composite Z-Scores (Depression Cohort; "
        f"N_control={n_controls_total}, N_depression={n_depressed_total})"
    ),
    save_path=domain_plot_path,
)
print(f"  Domain z-score plot saved : {domain_plot_path}")

# ---- Save z-scored data as CSV ----
output_csv = os.path.join(OUTPUT_DIR, f"depression_cohort_z_scores_{DEPRESSION_CODES}.csv")
z_scored_data.to_csv(output_csv, index=False)
print(f"\nZ-scored dataset saved to  : {output_csv}")
print(f"  Shape: {z_scored_data.shape}")

## Step 6: Per-Cluster Cognitive Profiles

This is the main analysis loop. It iterates over all **6 connectivity modality × direction combinations**:

| `conn_type` | `dir_type` | Cluster column |
|---|---|---|
| `functional` | `internal` | `functional_internal_cluster` |
| `functional` | `external` | `functional_external_cluster` |
| `structural` | `internal` | `structural_internal_cluster` |
| `structural` | `external` | `structural_external_cluster` |
| `sfc` | `internal` | `sfc_internal_cluster` |
| `sfc` | `external` | `sfc_external_cluster` |

For each combination, **two clusters (0 and 1)** of depressed subjects are analysed. Within each cluster, Steps 3–5 are repeated (z-scores → QR tests → FDR → visualizations), using the control group as reference.

At the end of each modality × direction loop, a **between-cluster comparison** is run:
- The z-scored rows for Cluster 0 and Cluster 1 are concatenated into a single temporary dataframe.
- Between-group quantile regression (`test_against_zero=False`) is run with **Cluster 1 as the reference group**, testing whether Cluster 0 differs at the median.
- FDR correction is applied, and the corrected p-values are registered via `register_radar_overlay_significance` so they can be shown as significance markers on radar overlay plots.

### Cluster label normalisation

Cluster column values may arrive as integers or floats (`0`, `1`, `0.0`, …) or as strings (`"Cluster 0"`, `"Cluster 1"`). The helper below canonicalises them to the string form `"Cluster 0"` / `"Cluster 1"` so all downstream code uses a consistent label.

In [ ]:
def _normalize_cluster_label(value):
    """Canonicalise a cluster column value to the string 'Cluster 0' or 'Cluster 1'."""
    if pd.isna(value):
        return value
    value_str = str(value).strip()
    if value_str.lower().startswith("cluster"):
        return value_str  # already in canonical form
    try:
        return f"Cluster {int(float(value_str))}"
    except (ValueError, TypeError):
        return value_str

# Quick smoke test
assert _normalize_cluster_label(0)      == "Cluster 0"
assert _normalize_cluster_label("1")    == "Cluster 1"
assert _normalize_cluster_label("0.0")  == "Cluster 0"
assert _normalize_cluster_label("Cluster 1") == "Cluster 1"
print("_normalize_cluster_label OK")

### 6a: Within-Cluster Analysis (Steps 3–5 repeated per cluster)

For each cluster subset (controls + one cluster of depressed subjects):
1. Compute robust z-scores referenced to controls.
2. Reverse latency/error task z-scores.
3. Run one-sample median QR tests; apply FDR (task level).
4. Compute domain composites; run one-sample median QR tests; apply FDR (domain level).
5. Save per-cluster radar plots (task + domain) to the `reports/plots/schaefer1000+tian54/<conn>_con/` subdirectory.
6. Collect the **depressed-only z-scored rows** into `cluster_between_parts` for the between-cluster comparison in Step 6b.

In [ ]:
print("[6/6] Per-cluster cognitive dysfunction profiles...\n")

# Outer loops: all 6 modality × direction combinations
for conn_type in ["functional", "structural", "sfc"]:
    for dir_type in ["internal", "external"]:
        cluster_col = f"{conn_type}_{dir_type}_cluster"

        if cluster_col not in data.columns:
            print(f"  [SKIP] '{cluster_col}' not found in dataset.")
            continue

        print(f"\n{'='*70}")
        print(f"  Connectivity type: {conn_type.upper()}  |  Direction: {dir_type.upper()}")
        print(f"{'='*70}")

        # Collect per-cluster z-scored depressed rows for later between-cluster QR
        cluster_between_parts = []

        for cluster_label in ["0", "1"]:
            cluster_name = f"Cluster {cluster_label}"
            print(f"\n  --- {cluster_name} vs controls ---")

            # Subset: all controls + depressed subjects assigned to this cluster
            cluster_subset = data[
                (data[GROUP_COLUMN] == 0)
                | ((data[GROUP_COLUMN] == 1) & (data[cluster_col] == cluster_label))
            ].copy()

            cluster_subset["Cluster"]           = cluster_subset[cluster_col].apply(_normalize_cluster_label)
            cluster_subset["Connectivity_Type"] = conn_type
            cluster_subset["Direction"]         = dir_type

            n_controls_cl  = int((cluster_subset[GROUP_COLUMN] == 0).sum())
            n_depressed_cl = int((cluster_subset[GROUP_COLUMN] == 1).sum())
            print(f"    N controls : {n_controls_cl}   N depressed ({cluster_name}): {n_depressed_cl}")

            # ---- Step 3 (within-cluster): task-wise robust z-scores ----
            unique_z_vars_cl = sorted(set(v for vals in DOMAINS.values() for v in vals))
            z_scored_cl = calculate_robust_z_scores(
                cluster_subset,
                vars=unique_z_vars_cl,
                group_column=GROUP_COLUMN,
                control_value=0,
                depression_value=1,
                log_path=ROBUST_Z_LOG_PATH,
                log_context=(
                    f"Within-cluster; connectivity_type: {conn_type}; "
                    f"direction: {dir_type}; cluster: {cluster_name}"
                ),
            )
            for var in VARS_TO_REVERSE:
                z_scored_cl[f"{var}_z"] = -z_scored_cl[f"{var}_z"]

            z_scored_cl.to_csv("/tmp/data.csv", index=False)
            p_vars_cl = quantile_regression(
                tmp_csv_path="/tmp/data.csv",
                dependent_variables=[f"{v}_z" for v in unique_z_vars_cl],
                covariates=COVARIATES,
                group_column=None,
                tau=0.5,
                R=QUANTILE_REGRESSION_BOOTSTRAP_R,
                test_against_zero=True,
                return_effects=False,
                r_output_log_path=R_LOG_PATH,
            )
            apply_multiple_testing_correction(
                p_values=list(p_vars_cl.values()),
                variable_names=[f"{v}_z" for v in unique_z_vars_cl],
                test_methods=["Quantile Regression"] * len(unique_z_vars_cl),
                method="fdr_bh",
                alpha=0.05,
                log_path=MT_LOG_PATH,
                log_context=(
                    f"Within-cluster; connectivity_type: {conn_type}; direction: {dir_type}; "
                    f"cluster: {cluster_name}; level: task-wise"
                ),
            )

            # ---- Step 4 (within-cluster): domain composites ----
            for domain, vars_list in DOMAINS.items():
                z_scored_cl = calculate_composite_z_score(
                    z_scored_cl,
                    z_vars=[f"{v}_z" for v in vars_list],
                    output_column=domain,
                    method="median",
                    log_path=COMPOSITE_Z_LOG_PATH,
                    log_context=(
                        f"Within-cluster; connectivity_type: {conn_type}; "
                        f"direction: {dir_type}; cluster: {cluster_name}; domain: {domain}"
                    ),
                )

            z_scored_cl.to_csv("/tmp/data.csv", index=False)
            p_dom_cl = quantile_regression(
                tmp_csv_path="/tmp/data.csv",
                dependent_variables=list(DOMAINS.keys()),
                covariates=COVARIATES,
                group_column=None,
                tau=0.5,
                R=QUANTILE_REGRESSION_BOOTSTRAP_R,
                test_against_zero=True,
                return_effects=False,
                r_output_log_path=R_LOG_PATH,
            )
            apply_multiple_testing_correction(
                p_values=list(p_dom_cl.values()),
                variable_names=list(DOMAINS.keys()),
                test_methods=["Quantile Regression"] * len(DOMAINS),
                method="fdr_bh",
                alpha=0.05,
                log_path=MT_LOG_PATH,
                log_context=(
                    f"Within-cluster; connectivity_type: {conn_type}; direction: {dir_type}; "
                    f"cluster: {cluster_name}; level: domain-wise"
                ),
            )

            task_z_vars_cl = [f"{v}_z" for v in unique_z_vars_cl]
            domain_vars_cl = list(DOMAINS.keys())

            # Collect depressed-only rows for between-cluster comparison (Step 6b)
            between_cols = (
                COVARIATES
                + ["Cluster", "Connectivity_Type", "Direction"]
                + [f"X{i}_0_{dir_type}_{conn_type}" for i in range(1, 10)]
                + task_z_vars_cl
                + domain_vars_cl
            )
            between_cols = [c for c in between_cols if c in z_scored_cl.columns]
            cluster_between_parts.append(
                z_scored_cl.loc[z_scored_cl[GROUP_COLUMN] == 1, between_cols].copy()
            )

            # ---- Step 5 (within-cluster): visualizations ----
            base_plot_dir = (
                f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con"
            )
            os.makedirs(base_plot_dir, exist_ok=True)

            plot_z_scores(
                z_scored_cl,
                z_vars=task_z_vars_cl,
                association_type=ASSOCIATION_with,
                type_z_score=f"task_{cluster_name.replace(' ', '_')}",
                overall_title=(
                    f"Task Z-Scores (Depression Cohort, {cluster_name}; "
                    f"N_control={n_controls_cl}, N_depression={n_depressed_cl})"
                ),
                save_path=(
                    f"{base_plot_dir}/modular_{DEPRESSION_CODES}_{ASSOCIATION_with}"
                    f"_task_z_scores_{cluster_name.replace(' ', '_')}_{dir_type}.png"
                ),
            )

            plot_z_scores(
                z_scored_cl,
                z_vars=domain_vars_cl,
                association_type=ASSOCIATION_with,
                type_z_score=f"domain_{cluster_name.replace(' ', '_')}",
                overall_title=(
                    f"Domain Composite Z-Scores (Depression Cohort, {cluster_name}; "
                    f"N_control={n_controls_cl}, N_depression={n_depressed_cl})"
                ),
                save_path=(
                    f"{base_plot_dir}/modular_{DEPRESSION_CODES}_{ASSOCIATION_with}"
                    f"_domain_z_scores_{cluster_name.replace(' ', '_')}_{dir_type}.png"
                ),
            )

            print(f"    Plots saved to {base_plot_dir}/")

        # ----------------------------------------------------------------
        # Step 6b: Between-cluster comparisons (Cluster 0 vs Cluster 1)
        # ----------------------------------------------------------------
        if len(cluster_between_parts) < 2:
            print("  Skipping between-cluster analysis (one or more clusters missing).")
            continue

        between_df   = pd.concat(cluster_between_parts, ignore_index=True)
        between_path = f"/tmp/{DEPRESSION_CODES}_{conn_type}_{dir_type}_cluster_zscores_concatenated.csv"
        between_df.to_csv(between_path, index=False)
        print(f"\n  Between-cluster dataset saved: {between_path}  (shape {between_df.shape})")

        # Connectivity vs cognitive z-score scatter plots (per module, task-level)
        for mod in [f"X{i}" for i in range(1, 10)]:
            conn_var = f"{mod}_0_{dir_type}_{conn_type}"
            if conn_var not in between_df.columns:
                continue
            plot_conn_cognition_association(
                data=between_df,
                connectivity_var=conn_var,
                cognitive_vars=[f"{v}_z" for v in unique_z_vars_cl],
                group_column="Cluster",
                save_path=(
                    f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/"
                    f"modular_{mod}_{dir_type}_{DEPRESSION_CODES}_{ASSOCIATION_with}_task_connectivity.png"
                ),
                overall_title=(
                    f"{mod} {conn_type.capitalize()} {dir_type.capitalize()} "
                    "Connectivity vs Cognitive Z-Scores"
                ),
            )

        # Task-level violin plots (Cluster 0 vs Cluster 1)
        try:
            plot_cognitive_distributions_violin(
                data=between_df,
                variables=[f"{v}_z" for v in unique_z_vars_cl],
                group_column="Cluster",
                plot_depression_clusters=True,
                cluster_column="Cluster",
                conn_type=conn_type,
                dir_type=dir_type,
                save_path=(
                    f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/"
                    f"modular_{DEPRESSION_CODES}_{ASSOCIATION_with}_task_clusters_violin_{dir_type}.png"
                ),
                title=f"Task z-score distributions by Cluster ({conn_type} {dir_type})",
            )
        except Exception as e:
            print(f"  Warning — task cluster violin failed: {e}")

        # Domain composite violin plots (Cluster 0 vs Cluster 1)
        try:
            plot_cognitive_distributions_violin(
                data=between_df,
                variables=list(DOMAINS.keys()),
                group_column="Cluster",
                plot_depression_clusters=True,
                cluster_column="Cluster",
                conn_type=conn_type,
                dir_type=dir_type,
                save_path=(
                    f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/"
                    f"modular_{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_clusters_violin_{dir_type}.png"
                ),
                title=f"Domain composite distributions by Cluster ({conn_type} {dir_type})",
            )
        except Exception as e:
            print(f"  Warning — domain cluster violin failed: {e}")

        # Between-cluster QR + FDR — task level
        p_vars_cls = quantile_regression(
            tmp_csv_path=between_path,
            dependent_variables=[f"{v}_z" for v in unique_z_vars_cl],
            covariates=COVARIATES,
            group_column="Cluster",
            reference_group="Cluster 1",
            comparison_groups=("Cluster 0",),
            test_against_zero=False,
            return_effects=False,
            tau=0.5,
            R=QUANTILE_REGRESSION_BOOTSTRAP_R,
            r_output_log_path=R_LOG_PATH,
        )
        reject_vars_cls, pvals_vars_cls = apply_multiple_testing_correction(
            p_values=list(p_vars_cls.values()),
            variable_names=[f"{v}_z" for v in unique_z_vars_cl],
            test_methods=["Quantile Regression"] * len(unique_z_vars_cl),
            method="fdr_bh",
            alpha=0.05,
            log_path=MT_LOG_PATH,
            log_context=(
                f"Between-cluster; connectivity_type: {conn_type}; direction: {dir_type}; "
                "clusters: Cluster 1 (ref) vs Cluster 0; level: task-wise"
            ),
        )
        register_radar_overlay_significance(
            depression_codes=DEPRESSION_CODES,
            association_type=ASSOCIATION_with,
            base_kind="task",
            conn_type=conn_type,
            dir_type=dir_type,
            variable_names=[f"{v}_z" for v in unique_z_vars_cl],
            pvals_corrected=pvals_vars_cls,
            comparison_label="Cluster 0 vs Cluster 1 (between-cluster, FDR)",
        )

        # Connectivity vs cognitive z-score scatter plots (per module, domain-level)
        for mod in [f"X{i}" for i in range(1, 10)]:
            conn_var = f"{mod}_0_{dir_type}_{conn_type}"
            if conn_var not in between_df.columns:
                continue
            plot_conn_cognition_association(
                data=between_df,
                connectivity_var=conn_var,
                cognitive_vars=list(DOMAINS.keys()),
                group_column="Cluster",
                save_path=(
                    f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con/"
                    f"modular_{mod}_{dir_type}_{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_connectivity.png"
                ),
                overall_title=(
                    f"{mod} {conn_type.capitalize()} {dir_type.capitalize()} "
                    "Connectivity vs Cognitive Z-Scores"
                ),
            )

        # Between-cluster QR + FDR — domain level
        p_dom_cls = quantile_regression(
            tmp_csv_path=between_path,
            dependent_variables=list(DOMAINS.keys()),
            covariates=COVARIATES,
            group_column="Cluster",
            reference_group="Cluster 1",
            comparison_groups=("Cluster 0",),
            test_against_zero=False,
            return_effects=False,
            tau=0.5,
            R=QUANTILE_REGRESSION_BOOTSTRAP_R,
            r_output_log_path=R_LOG_PATH,
        )
        reject_dom_cls, pvals_dom_cls = apply_multiple_testing_correction(
            p_values=list(p_dom_cls.values()),
            variable_names=list(DOMAINS.keys()),
            test_methods=["Quantile Regression"] * len(DOMAINS),
            method="fdr_bh",
            alpha=0.05,
            log_path=MT_LOG_PATH,
            log_context=(
                f"Between-cluster; connectivity_type: {conn_type}; direction: {dir_type}; "
                "clusters: Cluster 1 (ref) vs Cluster 0; level: domain-wise"
            ),
        )
        register_radar_overlay_significance(
            depression_codes=DEPRESSION_CODES,
            association_type=ASSOCIATION_with,
            base_kind="domain",
            conn_type=conn_type,
            dir_type=dir_type,
            variable_names=list(DOMAINS.keys()),
            pvals_corrected=pvals_dom_cls,
            comparison_label="Cluster 0 vs Cluster 1 (between-cluster, FDR)",
        )

        print(f"\n  Between-cluster QR + radar registry complete for {conn_type} {dir_type}.")

## Summary of Outputs

All outputs are written during the analysis. The cell below lists the expected paths so you can verify files exist and open them quickly.

In [ ]:
print("=" * 70)
print("Analysis complete — expected output files:")
print("=" * 70)

# ---- Text logs ----
print("\n[Text logs]")
for label, path in [
    ("R console / model summaries    ", R_LOG_PATH),
    ("Multiple testing corrections   ", MT_LOG_PATH),
    ("Robust z-score computation     ", ROBUST_Z_LOG_PATH),
    ("Composite z-score computation  ", COMPOSITE_Z_LOG_PATH),
]:
    exists = "✓" if os.path.exists(path) else "✗ (not yet created)"
    print(f"  {label}: {path}  [{exists}]")

# ---- CSV ----
print("\n[CSV]")
output_csv = os.path.join(OUTPUT_DIR, f"depression_cohort_z_scores_{DEPRESSION_CODES}.csv")
exists = "✓" if os.path.exists(output_csv) else "✗"
print(f"  Depression cohort z-scores     : {output_csv}  [{exists}]")

# ---- Overall cohort figures ----
print("\n[Figures — overall cohort]")
for fig_label, fig_path in [
    ("Task z-score radar  ", f"{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_task_z_scores.png"),
    ("Domain z-score radar", f"{PLOTS_DIR}/{DEPRESSION_CODES}_{ASSOCIATION_with}_domain_z_scores.png"),
]:
    exists = "✓" if os.path.exists(fig_path) else "✗"
    print(f"  {fig_label}: {fig_path}  [{exists}]")

# ---- Per-cluster figures ----
print("\n[Figures — per cluster]")
for conn_type in ["functional", "structural", "sfc"]:
    for dir_type in ["internal", "external"]:
        base = f"{PLOTS_DIR}/schaefer1000+tian54/{conn_type}_con"
        for cluster_label in ["0", "1"]:
            cluster_name = f"Cluster_{cluster_label}"
            for kind in ["task", "domain"]:
                p = f"{base}/modular_{DEPRESSION_CODES}_{ASSOCIATION_with}_{kind}_z_scores_{cluster_name}_{dir_type}.png"
                exists = "✓" if os.path.exists(p) else "✗"
                print(f"  [{exists}] {p}")
        for kind in ["task", "domain"]:
            p = f"{base}/modular_{DEPRESSION_CODES}_{ASSOCIATION_with}_{kind}_clusters_violin_{dir_type}.png"
            exists = "✓" if os.path.exists(p) else "✗"
            print(f"  [{exists}] {p}")

print("\n✓ = file present  |  ✗ = not yet written (run cells above first)")